# 02. Dataclasses

**Tier:** Intermediate. **Why you'd use this:** An item with a name and quantity needs a readable repr and value equality without hand-writing boilerplate. `@dataclass` generates that.

**After this lesson you can:** write a dataclass; rely on generated `__init__`, `__repr__`, and `__eq__`; explain what `frozen=True` adds for hashing.
About 25 minutes. **Uses:** [Classes](01-classes-lesson.ipynb)


## Quick review

1. Why is a mutable list on the class shared by every instance?
2. What does lookup put in `self` when you call a method on an instance?

<details><summary>Answers</summary>

It is one class attribute. The instance.
</details>


## The idea

`@dataclass` reads annotated fields and writes an `__init__`, a readable `__repr__`, and an `__eq__` that compares field values. `frozen=True` makes instances immutable after init and can make them hashable when fields are hashable. You still write methods for behaviour the generators do not cover.

<details><summary>Explain more</summary>

`Item("tea", 2)` works because `__init__` was generated. Two items with the same fields compare equal even if they are not the same object.
</details>


## See it work

Define an item and compare two with the same fields.


In [1]:
from dataclasses import dataclass


@dataclass
class Item:
    name: str
    qty: int


left = Item("tea", 2)
right = Item("tea", 2)
print(left)
print(left == right)
print(left is right)


Item(name='tea', qty=2)
True
False


## Check yourself

Answer these before you run the cell.

1. What methods does `@dataclass` generate by default?
2. Why can a frozen dataclass be a dict key when a normal one usually cannot?


In [2]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Item:
    name: str
    qty: int


item = Item("tea", 2)
print(item)
print({item: "ok"}[item])


Item(name='tea', qty=2)
ok


<details><summary>Answers</summary>

`__init__`, `__repr__`, and `__eq__`. Frozen instances can get `__hash__` when fields are hashable.
</details>


## Why it works this way

A dataclass is ordinary class machinery plus a decorator that fills in repetitive dunders from the annotated fields. The generated `__init__` assigns each field to `self`. The generated `__repr__` shows the class name and field values so debugging prints stay readable. The generated `__eq__` compares fields in order, which is value equality rather than identity. You still own any methods you write by hand. Deep dive: [Attribute lookup](../../internals/08-attributes/01-lookup-lesson.ipynb).

`frozen=True` blocks attribute assignment after init by raising on writes. That immutability is what makes a safe `__hash__` possible: hashable objects must not change while they sit in a set or dict. Mutable dataclasses keep `__hash__ = None` so they cannot silently break a set after a field changes. That is the same hash contract you met with lists versus tuples.

Cost is a bit of generation at import time and the usual per-instance storage. The design goal is less boilerplate for "bundle of fields" types, not a replacement for every class. When behaviour dominates data, or when `__init__` must enforce invariants with branches, a hand-written class can still be clearer than fighting the generator.

Evidence: equal items with different identities, a frozen item used as a dict key, and a failed mutation on a frozen instance. Construct fields from annotations in order. Defaults on later fields follow the same rules as function defaults. Reach for `dataclass` when the type is mostly data. Reach for a plain class when the type is mostly behaviour.


In [3]:
from dataclasses import dataclass


@dataclass
class Item:
    name: str
    qty: int


left = Item("tea", 2)
right = Item("tea", 2)
print(left)
print(left == right)
print(left is right)


Item(name='tea', qty=2)
True
False


In [4]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Item:
    name: str
    qty: int


item = Item("tea", 2)
item.qty = 3


FrozenInstanceError: cannot assign to field 'qty'

## When to use it

Use a dataclass for simple data records. Use `frozen=True` when instances should be hashable keys or stay unchanged. Write a normal class when setup logic needs a custom `__init__`. Don't mutate a dataclass you already put in a set. Don't assume every dataclass is hashable.


## Common mistake

Calling `hash` on a mutable dataclass, or mutating one after using it as a key.


In [5]:
from dataclasses import dataclass


@dataclass
class Item:
    name: str
    qty: int


item = Item("tea", 2)
print(hash(item))


TypeError: unhashable type: 'Item'

Freeze the dataclass when it must be a key, and keep fields hashable.


In [6]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Item:
    name: str
    qty: int


item = Item("tea", 2)
print(hash(item))
print({item: "ok"}[item])


6376997413366412531
ok


## Try it

Add a field `unit: str = "box"` to `Item`. Which generated method changes, and what does the repr print for `Item("tea", 2)`?


## Recap

- `@dataclass` generates `__init__`, `__repr__`, and `__eq__`.
- Equality is by field values.
- `frozen=True` enables hashing when fields are hashable.

**Next:** [practice](02-dataclasses-practice.ipynb)
